# CreditSense — Model Experiments

This notebook walks through:
1. Exploratory data analysis on the synthetic loan dataset
2. Feature importance comparison (XGBoost gain vs SHAP)
3. SHAP summary plots and individual explanations
4. Model calibration check
5. Threshold selection for risk bands


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, calibration_curve
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
ACCENT = '#B8FF3C'
print('Libraries loaded.')

## 1. Load & Explore Data

In [ ]:
# Run backend/data/generate.py first if this file doesn't exist
df = pd.read_csv('../backend/data/loan_data.csv')
print(f'Shape: {df.shape}')
print(f'Default rate: {df.defaulted.mean():.2%}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

key_features = [
    'cibil_score', 'debt_to_income_ratio', 'emi_to_income_ratio',
    'monthly_income', 'payment_timing_score', 'partial_payment_ratio',
    'inquiries_last_6m', 'existing_loans'
]

for ax, feat in zip(axes, key_features):
    for label, color in [(0, '#3B82F6'), (1, '#EF4444')]:
        subset = df[df.defaulted == label][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=color,
                label='Non-default' if label == 0 else 'Default', density=True)
    ax.set_title(feat, color='white', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_yticks([])

plt.suptitle('Feature Distributions by Default Status', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../docs/screenshots/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.drop('defaulted', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', color='white', pad=20)
plt.tight_layout()
plt.savefig('../docs/screenshots/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Train Model

In [ ]:
FEATURES = [
    'age', 'monthly_income', 'cibil_score', 'loan_amount',
    'loan_tenure_months', 'emi_to_income_ratio', 'existing_loans',
    'debt_to_income_ratio', 'employment_type', 'years_employed',
    'payment_timing_score', 'partial_payment_ratio',
    'months_since_delinquency', 'inquiries_last_6m'
]

X = df[FEATURES]
y = df['defaulted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=7,
    tree_method='hist', random_state=42, verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

probs = model.predict_proba(X_test)[:, 1]
print(f'AUROC: {roc_auc_score(y_test, probs):.4f}')
print(f'AUPRC: {average_precision_score(y_test, probs):.4f}')

## 3. SHAP Analysis

In [ ]:
# Compute SHAP values on test set sample
sample = X_test.sample(n=500, random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

# Summary plot — shows global feature importance + direction
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, sample, feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Feature Impact on Default Probability', pad=15)
plt.tight_layout()
plt.savefig('../docs/screenshots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Higher SHAP value = pushes prediction toward default')

In [ ]:
# Single prediction waterfall — pick a HIGH risk applicant
high_risk_idx = np.where(probs > 0.5)[0][0]
explanation = explainer(X_test.iloc[[high_risk_idx]])
shap.plots.waterfall(explanation[0], show=False)
plt.title(f'Waterfall — High Risk Applicant (p={probs[high_risk_idx]:.2%})', pad=15)
plt.tight_layout()
plt.savefig('../docs/screenshots/shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Calibration

In [ ]:
# Calibration curve — are predicted probabilities trustworthy?
# A well-calibrated model: P(default | predicted=0.3) ≈ 30% of those actually defaulted
fraction_pos, mean_pred = calibration_curve(y_test, probs, n_bins=15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'w--', alpha=0.4, label='Perfectly calibrated')
ax.plot(mean_pred, fraction_pos, color=ACCENT, lw=2, marker='o', ms=5, label='CreditSense')
ax.fill_between(mean_pred, fraction_pos, mean_pred,
                alpha=0.15, color=ACCENT)
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (actual defaults)')
ax.set_title('Model Calibration Curve', color='white')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../docs/screenshots/calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Close to diagonal = well calibrated. Probabilities can be trusted as actual risk estimates.')

## 5. Risk Band Threshold Analysis

In [ ]:
# What % of actual defaulters fall in each band?
thresholds = [0.15, 0.35, 0.60]
bands = pd.cut(probs, bins=[0] + thresholds + [1],
               labels=['LOW', 'MODERATE', 'HIGH', 'CRITICAL'])

results = pd.DataFrame({'band': bands, 'actual': y_test.values})
summary = results.groupby('band').agg(
    count=('actual', 'size'),
    defaults=('actual', 'sum'),
    default_rate=('actual', 'mean')
).round(3)

print('Risk Band Summary (test set):')
print(summary.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#22C55E', '#F59E0B', '#EF4444', '#7C3AED']
bars = ax.bar(summary.index, summary['default_rate'], color=colors, alpha=0.85, width=0.5)
ax.axhline(y_test.mean(), color='white', linestyle='--', alpha=0.5, label=f'Base rate ({y_test.mean():.1%})')
for bar, rate in zip(bars, summary['default_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=12, color='white')
ax.set_ylabel('Actual Default Rate in Band')
ax.set_title('Risk Band Discrimination', color='white')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../docs/screenshots/risk_band_discrimination.png', dpi=150, bbox_inches='tight')
plt.show()